## **SNAP Jupyter demo notebook**
**SAR speckle filter showcase — visual and quantitative comparison**

In summary, this workflow contains:

- Background on speckle and why SAR images need filtering
- Preprocess a single Sentinel-1 GRD scene (calibration + terrain correction)
- Apply five common single-channel speckle filters: Boxcar, Frost, Refined Lee, IDAN, Lee Sigma
- Visual side-by-side grid + quantitative comparison via **Equivalent Number of Looks (ENL)** measured over a homogeneous patch

Complexity: intermediate

##### ***About the test data:***

Any Sentinel-1 GRD product covering an area with both a homogeneous patch (open water, bare field, or grassland) and structured features (urban, forest edges, roads) is ideal — the homogeneous patch is used for the ENL metric, the structured regions for visually assessing edge preservation.

Filename pattern: `S1*_IW_GRDH_1S*` from [Copernicus Browser](https://dataspace.copernicus.eu/browser/).

Place the `.SAFE` directory under `data/` and update the path variable below.

##### ***Some information on the Python environment:***

In [ ]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

##### ***Import Python packages...***

In [ ]:
import esa_snappy
from esa_snappy import ProductIO

import snapista
from snapista import Graph
from snapista import Operator

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Plot helpers (with ENL):***

In [ ]:
def _read_band(product, name):
    band = product.getBand(name)
    if band is None:
        raise KeyError(f"Band {name!r} not found. Available: {[b.getName() for b in product.getBands()]}")
    w, h = band.getRasterWidth(), band.getRasterHeight()
    data = np.zeros(w * h, np.float32)
    band.readPixels(0, 0, w, h, data)
    data.shape = h, w
    return data

def plot_grid(images, titles, suptitle=None, cmap='gray', vmin=None, vmax=None, ncols=3):
    """Plot a list of 2D arrays in a grid."""
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
    axs = np.array(axs).reshape(-1)
    for ax, img, title in zip(axs, images, titles):
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axs[len(images):]:
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

def compute_enl(data, x0, y0, w, h):
    """Equivalent Number of Looks over a (presumably homogeneous) patch."""
    patch = data[y0:y0 + h, x0:x0 + w]
    mean = np.nanmean(patch)
    var = np.nanvar(patch)
    return float(mean * mean / var) if var > 0 else float('inf')

def find_band(product, *patterns):
    names = [b.getName() for b in product.getBands()]
    for pat in patterns:
        for n in names:
            if pat.lower() in n.lower():
                return n
    raise KeyError(f"No band matching {patterns!r} found. Available: {names}")

---

### ***Background: SAR speckle***

SAR images look noisy. That "noise" is **speckle** — it's not sensor noise but interference: many independent scatterers inside each resolution cell each contribute their own complex amplitude, and the random sum gives multiplicative, Rayleigh/Gamma-distributed fluctuations. Speckle is a *signal*, not an *error*, but it makes interpretation and segmentation hard.

Filters reduce speckle by averaging over neighbouring pixels, with various strategies for preserving edges and point targets:

| Filter | Idea | Edge preservation |
|:-------|:-----|:------------------|
| **Boxcar** | uniform mean over a window | poor — blurs everything |
| **Frost** | exponentially weighted by local variation | moderate |
| **Refined Lee** | direction-aware: pick a sub-window aligned with local gradient | good |
| **IDAN** | grow an irregular region of similar pixels around each centre | very good |
| **Lee Sigma** | mean of pixels within ±σ of the centre | good, clips outliers |

We'll quantify *speckle reduction* with the **Equivalent Number of Looks** (ENL):

$$\mathrm{ENL} = \frac{\mu^2}{\sigma^2}$$

measured on a homogeneous (uniform-reflectance) patch. Higher ENL = more speckle smoothed out. Single-look GRD has ENL ≈ 1; a 5×5 boxcar boosts that toward 25 if the patch is truly homogeneous; smart filters trade some ENL for edge preservation.

---

### ***Configure input paths***

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

grd_product = os.path.join(data_dir, 'S1A_IW_GRDH_1SDV_<acquisition>.SAFE', 'manifest.safe')
polarisation = 'VV'

# Crop to a small AOI for tutorial speed: x, y, w, h in source pixels
aoi = {'x': 0, 'y': 0, 'w': 2000, 'h': 2000}

# Homogeneous patch in the AOI for ENL computation (in AOI-local coords)
enl_patch = {'x': 100, 'y': 100, 'w': 200, 'h': 200}

---
## ***Part 1 — Preprocess the GRD scene***
---

Standard GRD prep: orbit, thermal-noise removal, border-noise removal, calibration to sigma-nought, and a subset crop. We skip terrain correction for now so the filters operate in the original GRD geometry.

In [ ]:
g_prep = Graph()
g_prep.add_node(operator=Operator('Read', file=grd_product), node_id='Read')
g_prep.add_node(operator=Operator('Apply-Orbit-File',
                                  orbitType='Sentinel Precise (Auto Download)',
                                  continueOnFail='true'),
                node_id='Orbit', source='Read')
g_prep.add_node(operator=Operator('ThermalNoiseRemoval',
                                  selectedPolarisations=polarisation),
                node_id='ThermalNoise', source='Orbit')
g_prep.add_node(operator=Operator('Remove-GRD-Border-Noise',
                                  selectedPolarisations=polarisation),
                node_id='BorderNoise', source='ThermalNoise')
g_prep.add_node(operator=Operator('Calibration',
                                  outputSigmaBand='true',
                                  selectedPolarisations=polarisation),
                node_id='Calib', source='BorderNoise')
g_prep.add_node(operator=Operator('Subset',
                                  region=f"{aoi['x']},{aoi['y']},{aoi['w']},{aoi['h']}"),
                node_id='Subset', source='Calib')

prep_out = os.path.join(results_dir, 'snap_nb_speckle_prep.dim')
g_prep.add_node(operator=Operator('Write', file=prep_out, formatName='BEAM-DIMAP'),
                node_id='Write', source='Subset')

g_prep.save_graph(os.path.join(graphs_dir, 'snap_nb_speckle_prep.xml'))
g_prep.run()

---
## ***Part 2 — Apply five filters***
---

For each filter we run a one-node graph: `Read → Speckle-Filter → Write`. The input is the already-preprocessed product from Part 1; the only thing that changes between runs is the `filter` parameter.

In [ ]:
filters = [
    'Boxcar',
    'Frost',
    'Refined Lee',
    'IDAN',
    'Lee Sigma',
]

filter_outputs = {}
for f in filters:
    g = Graph()
    g.add_node(operator=Operator('Read', file=prep_out), node_id='Read')
    g.add_node(operator=Operator('Speckle-Filter',
                                 filter=f,
                                 windowSize='5x5'),
               node_id='Filter', source='Read')
    out = os.path.join(results_dir,
                       f"snap_nb_speckle_{f.replace(' ', '_').lower()}.dim")
    g.add_node(operator=Operator('Write', file=out, formatName='BEAM-DIMAP'),
               node_id='Write', source='Filter')
    g.run()
    filter_outputs[f] = out
    print(f'  done: {f}')
print('All filters done.')

---
## ***Part 3 — Visual comparison***
---

Plot the unfiltered baseline plus all five filtered results on a single grid, with a shared dB colour scale so the comparison is honest.

In [ ]:
# Read all products and convert the Sigma0 band to dB for display
def sigma0_db(product):
    name = find_band(product, 'Sigma0')
    data = _read_band(product, name)
    data = np.maximum(data, 1e-6)
    return 10.0 * np.log10(data)

p_prep = ProductIO.readProduct(prep_out)
images = [sigma0_db(p_prep)]
titles = ['unfiltered']
loaded_products = [p_prep]
for f in filters:
    p = ProductIO.readProduct(filter_outputs[f])
    loaded_products.append(p)
    images.append(sigma0_db(p))
    titles.append(f)

# Shared dB stretch across all panels
stack = np.stack(images)
vmin = np.nanpercentile(stack, 2)
vmax = np.nanpercentile(stack, 98)
plot_grid(images, titles, suptitle='Sigma0 [dB] — baseline + 5 filters',
          cmap='gray', vmin=vmin, vmax=vmax, ncols=3)

---
## ***Part 4 — Quantitative comparison: ENL on a homogeneous patch***
---

Compute ENL on the patch defined by `enl_patch` in the *Configure* cell. Choose a patch over open water or a uniform field for a meaningful comparison.

In [ ]:
# ENL is computed on linear-intensity Sigma0, not dB
def linear_sigma0(product):
    return _read_band(product, find_band(product, 'Sigma0'))

x0, y0, w, h = enl_patch['x'], enl_patch['y'], enl_patch['w'], enl_patch['h']
print(f'ENL patch: ({x0},{y0})  {w}x{h} pixels\n')
print(f"{'filter':<14s}  {'ENL':>10s}")
print('-' * 28)
enl_baseline = compute_enl(linear_sigma0(p_prep), x0, y0, w, h)
print(f"{'unfiltered':<14s}  {enl_baseline:>10.2f}")
for f, p in zip(filters, loaded_products[1:]):
    enl = compute_enl(linear_sigma0(p), x0, y0, w, h)
    print(f"{f:<14s}  {enl:>10.2f}")

for p in loaded_products:
    p.dispose()

---

### ***Summary***

What have we learnt in this notebook?

- Speckle is multiplicative interference, not sensor noise; raw GRD has ENL ≈ 1 and looks visually noisy.
- A 5×5 **Boxcar** filter gives the highest ENL on flat regions but smears edges. It's the upper bound on speckle reduction with a fixed-shape window.
- **Refined Lee** and **IDAN** trade some ENL for edge and point-target preservation — their ENL on a flat patch is lower than Boxcar but their visual results are sharper at boundaries.
- **Frost** sits in the middle; **Lee Sigma** is good when the scene has many isolated bright targets you want to keep.
- ENL on a homogeneous patch is the standard quantitative metric. For real applications, also measure **edge preservation** (e.g., FOM) and **point-target preservation** — a filter that maxes out ENL but blurs edges is rarely useful.